In [73]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
# from langchain_tavily import TavilySearch
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests
from langchain_groq import ChatGroq

In [74]:
# from langchain.agents import create_react_agent, AgentExecutor
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

In [75]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [76]:
search_tool = TavilySearchResults(max_results=2)

In [78]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [79]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-september-2026',
  'content': "Speed 0.75X 1X 1.5X 2X\n\n Read AI-generated summary \n\n   Check out the latest AI news from September 2026 in our monthly roundup.\n   Gemini 4 Argon leads our new frontier models with advanced reasoning for complex tasks.\n   We’ve launched natural voice models and new app integrations to simplify your daily workflow.\n   Exciting scientific breakthroughs include mapping human DNA and tracking global methane from space.\n   New tools like Googlebook and Gemini for Windows help you get more done effortlessly.\n\n Summaries were generated by Google AI. Generative AI is experimental. [...] September's headline belongs to Gemini 4 Argon, our newest frontier model. Built with advanced reasoning and a 1-million-token output limit, it’s designed to tackle complex challenges — especially in cybersecurity defense. The release capped an active month for Google AI launches that includ

In [80]:
# ==========================================
# LLM
# ==========================================

llm = ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0,
    api_key=GROQ_API_KEY
)

In [81]:
response = llm.invoke("Tell me a joke about AI")
response

/opt/anaconda3/envs/cvdev/lib/python3.11/site-packages/langchain_groq/chat_models.py:432: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  response = response.dict()


AIMessage(content="Sure, here's one for you:\n\nWhy did the AI go to therapy?\n\nBecause it kept over‑thinking its **neural** network and couldn't stop **re‑training** its feelings! 😄", response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 77, 'total_tokens': 142, 'completion_time': 0.13617891, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.015580676, 'prompt_tokens_details': None, 'queue_time': 0.222083901, 'total_time': 0.151759586}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'finish_reason': 'stop', 'logprobs': None}, id='run-264d7c51-6aa3-49f2-94d4-6dd8edf9efa6-0')

In [82]:
# ==========================================
# PROMPT
# ==========================================

# prompt = hub.pull("hwchase17/react")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the provided tools when needed, "
               "and search the web for anything about current events."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

In [84]:
prompt

ChatPromptTemplate(input_variables=['input'], input_types={'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, partial_variables={'agent_scratchpad': []}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful assistant. Use the provided tools when needed, and search the web for anything about current events.')), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), MessagesPlaceholder(variable_name='agent_scratchpad', optional=True)])

In [85]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [86]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [88]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [89]:
# ==========================================
# RUN
# ==========================================

# llm = ChatGroq(
#     model="llama-3.3-70b-versatile",
#     temperature=0,
#     api_key=GROQ_API_KEY,
# )

# agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
# agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

response = agent_executor.invoke({
    "input": "Find the latest reliable news about the US-Iran conflict."
})



> Entering new AgentExecutor chain...


/opt/anaconda3/envs/cvdev/lib/python3.11/site-packages/langchain_groq/chat_models.py:432: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  response = response.dict()



Invoking: `tavily_search_results_json` with `{'query': 'latest news US Iran conflict 2024 October'}`


[{'url': 'https://www.cfr.org/global-conflict-tracker/conflict/confrontation-between-united-states-and-iran', 'content': 'In 2024, the Israel-Iran confrontation shifted from proxy-based hostilities to direct exchanges of fire. In April, anIsraeli air strikeon an Iranian consulate annex in Damascus killed two Iranian generals, prompting Iran to launch over three hundreddrones and missiles at Israel—its first direct strike on Israeli territory. Following Israel’s killing of Hamas leader Yahya Sinwar and Hezbollah leader Hassan Nasrallah, Iran launched180 ballistic missilesagainst Israel in October 2024. Israel responded with itslargest-ever direct attackon Iran, targeting air defenses and missile production facilities. Coupled with the downfall of the Assad regime in Syria,the losses significantly degraded Iran’s axis of resistance. [...] Middle East and North Africa\n   Interstate War

/opt/anaconda3/envs/cvdev/lib/python3.11/site-packages/langchain_groq/chat_models.py:432: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  response = response.dict()



Invoking: `tavily_search_results_json` with `{'query': 'October 2024 US Iran tensions latest news Reuters October 2024 US Iran conflict'}`


[{'url': 'https://www.cfr.org/global-conflict-tracker/conflict/confrontation-between-united-states-and-iran', 'content': 'In 2024, the Israel-Iran confrontation shifted from proxy-based hostilities to direct exchanges of fire. In April, anIsraeli air strikeon an Iranian consulate annex in Damascus killed two Iranian generals, prompting Iran to launch over three hundreddrones and missiles at Israel—its first direct strike on Israeli territory. Following Israel’s killing of Hamas leader Yahya Sinwar and Hezbollah leader Hassan Nasrallah, Iran launched180 ballistic missilesagainst Israel in October 2024. Israel responded with itslargest-ever direct attackon Iran, targeting air defenses and missile production facilities. Coupled with the downfall of the Assad regime in Syria,the losses significantly degraded Iran’s axis of resistance. [...] Iran reta

...

/opt/anaconda3/envs/cvdev/lib/python3.11/site-packages/langchain_groq/chat_models.py:432: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  response = response.dict()



Invoking: `tavily_search_results_json` with `{'query': '2026 US Iran tensions Reuters March 2026', 'top_k': 10}`


[{'url': 'https://www.reuters.com/world/middle-east/top-us-general-will-take-time-achieve-objectives-iran-2026-03-02', 'content': 'the U.S. attack on Iran risked plunging the United States into a new, open-ended conflict in the Middle East, March 1, 2026.'}, {'url': 'https://www.reuters.com/world/middle-east/iran-conflict-poses-new-risk-us-economic-resilience-2026-03-02', 'content': '"A pillar of our 2026 outlook was the observed \'fading of caution\' regarding U.S. policy. \u200bEarly-year data suggested that businesses were moving past the paralysis in hiring and non-tech capex (capital expenditure) and beginning to deploy their resilient profits and capital," Joseph Lupton, an economist at JPMorgan, wrote in \u2060a note over the weekend after the U.S. bombardment of Iran had begun. "This nascent recovery is now at risk. A military war, layered on top of the ongoing U.

/opt/anaconda3/envs/cvdev/lib/python3.11/site-packages/langchain_groq/chat_models.py:432: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  response = response.dict()



Invoking: `tavily_search_results_json` with `{'query': 'November 2024 US Iran missile attack Reuters November 2024 US response', 'recency_days': 180, 'top_k': 10}`


[{'url': 'https://thesoufancenter.org/intelbrief-2026-october-29', 'content': 'Iranian leaders have suggested that senior officials would be legitimate targets as a response to the U.S. and Israeli strikes on senior Iranian officials both in 2025 and 2026. However, U.S. and Western security authorities have largely succeeded in excluding Iranian security personnel from their territories, and Iran would likely have to subcontract its attacks. In the past, purported Iranian assassination and other terrorist plots have relied on hapless non-Iranian operatives who have been arrested before they could do harm. [...] Placing little faith in the diplomacy, Iranian hardliners insist that Tehran leave no uncertainty about the scope, scale, and target set for Iran’s response if Trump attacks again after the U.S. midterms. Iranian l

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01k5srhjejeagr1f6d4aefzy2e` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8330, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [90]:
print(response["output"])

TypeError: 'AIMessage' object is not subscriptable